<a href="https://colab.research.google.com/github/orqidea-labs/3186242/blob/main/Ehercicio_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

datos que se tienen del wordk bank generar una app web que permita interactuar en un contexto pedagogico, que elementos quiero controlar facilmente, y pensar en un ejercicio pedagogico que apunten a la innovacion, comunicacion, administracion. optimizar el entrenmiento en alguna de los anteriores.

In [1]:
# ── PASO 1: CONFIGURACIÓN DE ENTORNO Y JSON DE CONTROL ──────────────────────

import os
import json
import pandas as pd
import requests
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Definición del "JSON" de configuración (Para no "quemar" valores)
# Este diccionario centraliza el comportamiento de la App.
config_app = {
    "api_settings": {
        "base_url": "https://api.worldbank.org/v2",
        "headers": {"Accept": "application/json"},
        "timeout": 30,
        "pause_sec": 0.1
    },
    "default_params": {
        "year": 2022,
        "max_indicators": 300,
        "output_csv": "wb_all_indicators.csv",
        "failed_csv": "wb_failed.csv"
    },
    "diagnose_indicators": {
        "SP.POP.TOTL": "Población total",
        "NY.GDP.PCAP.CD": "PIB per cápita (USD)",
        "SH.DYN.MORT": "Mortalidad infantil",
        "SE.ADT.LITR.ZS": "Tasa de alfabetización adultos (%)",
        "VC.IHR.PSRC.P5": "Homicidios por 100k hab"
    },
    "id_candidates": [
        "iso3c", "iso3", "iso_a3", "ISO_A3", "country_code",
        "country", "Country", "name", "Name"
    ]
}

# 2. Guardar configuración en un archivo físico para simular persistencia
with open('config_app.json', 'w') as f:
    json.dump(config_app, f, indent=4)

# 3. Función auxiliar para leer configuración (siempre del archivo)
def get_config():
    with open('config_app.json', 'r') as f:
        return json.load(f)

print("✅ Entorno configurado.")
print(f"✅ Archivo 'config_app.json' generado con {len(config_app['diagnose_indicators'])} indicadores de prueba.")

✅ Entorno configurado.
✅ Archivo 'config_app.json' generado con 5 indicadores de prueba.


In [3]:
# ── PASO 2: MOTOR DE COMUNICACIÓN CON LA API (CAPA DE DATOS) ────────────────

def wb_get_payload(url, params=None):
    """
    Consulta la API del World Bank usando la configuración dinámica.
    No hay valores "quemados": el timeout y los headers vienen del JSON.
    """
    # Recuperamos la configuración fresca
    cfg = get_config()["api_settings"]

    p = {"format": "json", "per_page": 1000, **(params or {})}

    for attempt in range(3):
        try:
            r = requests.get(
                url,
                params=p,
                headers=cfg["headers"],
                timeout=cfg["timeout"]
            )
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == 2:
                print(f"❌ Error crítico tras 3 intentos en {url}: {e}")
                return [None, None]
            time.sleep(2)

def get_country_list():
    """Obtiene la lista de países filtrando agregados regionales."""
    cfg = get_config()
    base = cfg["api_settings"]["base_url"]

    data = wb_get_payload(f"{base}/country", {"per_page": 300})

    if data[1]:
        countries = pd.DataFrame([
            {
                "iso3c": c["id"],
                "country": c["name"],
                "region": c.get("region", {}).get("value", "")
            }
            for c in data[1]
            if c.get("region", {}).get("id", "") != "NA"
        ])
        return countries
    return pd.DataFrame()

def get_indicator_catalog(topic_id=None):
    """Descarga el catálogo de indicadores, opcionalmente filtrado por tema."""
    cfg = get_config()
    base = cfg["api_settings"]["base_url"]

    url_ind = f"{base}/indicator"
    if topic_id:
        url_ind = f"{base}/topic/{topic_id}/indicator"

    data = wb_get_payload(url_ind)
    if not data[0]: return pd.DataFrame()

    total_pages = data[0]["pages"]
    records = data[1] or []

    # Nota: En una app real, limitaríamos esto para no saturar el Colab
    # Aquí procesamos las páginas necesarias
    for page in range(2, min(total_pages + 1, 5)): # Limitado a 5 páginas para velocidad
        chunk = wb_get_payload(url_ind, {"page": page})
        if chunk[1]:
            records += chunk[1]

    return pd.DataFrame([
        {
            "indicator_id": r["id"],
            "name": r["name"],
            "source": r.get("source", {}).get("value", ""),
            "topics": ", ".join(t.get("value","") for t in r.get("topics", []))
        }
        for r in records
    ])

print("✅ Motor de API listo.")
print("✅ Funciones modularizadas y vinculadas a config_app.json.")

✅ Motor de API listo.
✅ Funciones modularizadas y vinculadas a config_app.json.


In [4]:
# ── PASO 3: LÓGICA DE DIAGNÓSTICO Y CARGA INTELIGENTE ───────────────────────

def smart_load_data():
    """
    Decide la mejor fuente de datos sin valores 'quemados'.
    Prioridad: Memoria > CSV > Descarga de diagnóstico.
    """
    cfg = get_config()
    file_name = cfg["default_params"]["output_csv"]
    test_inds = cfg["diagnose_indicators"]
    base_url = cfg["api_settings"]["base_url"]

    # 1. Intentar recuperar de la sesión global (si existe df_wide)
    if 'df_wide' in globals() and df_wide is not None:
        if not df_wide.empty and len(df_wide.columns) > 2:
            print(f"💡 Datos recuperados de la memoria RAM: {df_wide.shape}")
            return df_wide

    # 2. Intentar cargar desde el archivo CSV
    if os.path.exists(file_name):
        df = pd.read_csv(file_name, encoding="utf-8-sig")
        if not df.empty and len(df.columns) > 2:
            print(f"💡 Datos cargados desde el archivo persistente '{file_name}': {df.shape}")
            return df

    # 3. Descarga de diagnóstico (Fallback)
    print("🚀 Iniciando descarga de diagnóstico (indicadores base)...")

    # Obtener lista de países usando el motor del Paso 2
    df_countries = get_country_list()
    if df_countries.empty:
        raise RuntimeError("No se pudo obtener la lista de países.")

    iso_str = ";".join(df_countries["iso3c"].tolist())
    frames = []

    for ind_id, ind_name in test_inds.items():
        url = f"{base_url}/country/{iso_str}/indicator/{ind_id}"
        # Usamos el motor de peticiones con reintentos del Paso 2
        data = wb_get_payload(url, {"date": cfg["default_params"]["year"]})

        if data and len(data) > 1 and data[1]:
            rows = [
                {
                    "iso3c": r["countryiso3code"],
                    "country": r["country"]["value"],
                    ind_name: r["value"]
                }
                for r in data[1]
                if r.get("countryiso3code") and r.get("value") is not None
            ]
            if rows:
                frames.append(pd.DataFrame(rows).set_index(["iso3c", "country"]))
                print(f"   ✅ {ind_name} sincronizado.")

        time.sleep(cfg["api_settings"]["pause_sec"])

    if frames:
        consolidated_df = pd.concat(frames, axis=1).reset_index()
        consolidated_df.to_csv(file_name, index=False, encoding="utf-8-sig")
        print(f"✅ Descarga completada. Total: {consolidated_df.shape}")
        return consolidated_df

    return pd.DataFrame()

# Ejecutar la carga inicial
df_wide = smart_load_data()

🚀 Iniciando descarga de diagnóstico (indicadores base)...
   ✅ Población total sincronizado.
   ✅ PIB per cápita (USD) sincronizado.
   ✅ Mortalidad infantil sincronizado.
   ✅ Tasa de alfabetización adultos (%) sincronizado.
   ✅ Homicidios por 100k hab sincronizado.
✅ Descarga completada. Total: (217, 7)


In [5]:
# ── PASO 4: INTERFAZ DE USUARIO (UI CON WIDGETS) ───────────────────────────

def create_ui(df):
    """Genera los controles interactivos basados en el DataFrame cargado."""
    cfg = get_config()

    # 1. Identificar columnas de datos (excluyendo IDs)
    id_cols = [c for c in df.columns if c in cfg["id_candidates"]]
    indicator_cols = [c for c in df.columns if c not in id_cols]

    # --- WIDGETS DE CONTROL ---

    # Selector de Indicador Principal
    w_indicator = widgets.Dropdown(
        options=sorted(indicator_cols) if indicator_cols else ["No hay datos"],
        description='📊 Indicador:',
        style={'description_width': 'initial'},
        layout={'width': 'max-content'}
    )

    # Selector de País para consulta rápida
    w_country = widgets.Combobox(
        placeholder='Escribe un país...',
        options=list(df['country'].unique()) if 'country' in df.columns else [],
        description='🌍 País:',
        ensure_option=True,
        style={'description_width': 'initial'}
    )

    # Slider para filtrar por cobertura de datos (mínimo % de datos que debe tener el país)
    w_coverage = widgets.FloatSlider(
        value=0,
        min=0,
        max=100.0,
        step=5,
        description='📉 Cobertura Mín %:',
        style={'description_width': 'initial'},
        orientation='horizontal'
    )

    # Botón de Acción
    w_btn_run = widgets.Button(
        description='Generar Consulta',
        button_style='success', # 'success', 'info', 'warning', 'danger' or ''
        icon='search'
    )

    # Contenedor de salida para los resultados
    output_panel = widgets.Output()

    # --- DISEÑO (LAYOUT) ---
    input_box = widgets.VBox([
        widgets.HTML("<h3>Panel de Control de Indicadores</h3>"),
        widgets.HBox([w_indicator, w_country]),
        widgets.HBox([w_coverage, w_btn_run]),
        widgets.HTML("<hr>")
    ])

    return input_box, output_panel, w_indicator, w_country, w_coverage, w_btn_run

# Inicializar la UI
ui_box, out_panel, drop_ind, combo_ctry, slider_cov, btn_run = create_ui(df_wide)

# Mostrar la interfaz
display(ui_box)
display(out_panel)

print("✅ Interfaz generada. Lista para vincular con el cerebro de la aplicación.")

Output()

✅ Interfaz generada. Lista para vincular con el cerebro de la aplicación.


In [6]:
# ── PASO 5: ORQUESTADOR DE VISUALIZACIÓN Y REPORTES ────────────────────────

def procesar_consulta(b):
    """Función que se ejecuta al hacer clic en 'Generar Consulta'."""
    with out_panel:
        clear_output(wait=True) # Limpiar el panel de resultados previos

        # 1. Obtener valores de los widgets
        indicador_sel = drop_ind.value
        pais_sel = combo_ctry.value
        cobertura_min = slider_cov.value
        cfg = get_config()

        if indicador_sel == "No hay datos":
            print("❌ No hay indicadores disponibles para analizar.")
            return

        # 2. Filtrado dinámico por cobertura
        # Calculamos qué % de indicadores tiene cada país
        id_cols = [c for c in df_wide.columns if c in cfg["id_candidates"]]
        cobertura_por_pais = df_wide.set_index(id_cols).notna().mean(axis=1) * 100
        paises_validos = cobertura_por_pais[cobertura_por_pais >= cobertura_min].index.get_level_values('country')
        df_filtrado = df_wide[df_wide['country'].isin(paises_validos)]

        # 3. Renderizado de Resultados
        print(f"📊 REPORTE DE CONSULTA: {indicador_sel.upper()}")
        print("="*60)

        # --- SECCIÓN A: ESTADÍSTICAS GLOBALES ---
        stats = df_filtrado[indicador_sel].describe()

        col1, col2, col3 = widgets.Output(), widgets.Output(), widgets.Output()
        display(widgets.HBox([col1, col2, col3]))

        with col1:
            print(f"📈 Resumen Estadístico:")
            print(f"   • Países con datos: {int(stats['count'])}")
            print(f"   • Promedio Global: {stats['mean']:,.2f}")
            print(f"   • Mediana: {stats['50%']:,.2f}")

        with col2:
            print(f"🚩 Valores Extremos:")
            print(f"   • Máximo: {stats['max']:,.2f}")
            print(f"   • Mínimo: {stats['min']:,.2f}")

        # --- SECCIÓN B: CONSULTA ESPECÍFICA DE PAÍS ---
        if pais_sel:
            dato_pais = df_wide[df_wide['country'] == pais_sel][indicador_sel].values
            print("\n" + "-"*60)
            if len(dato_pais) > 0 and not pd.isna(dato_pais[0]):
                posicion = (df_wide[indicador_sel] > dato_pais[0]).sum() + 1
                print(f"🌍 Resultado para {pais_sel.upper()}:")
                print(f"   Dato: {dato_pais[0]:,.2f} | Ranking Mundial: #{posicion}")
            else:
                print(f"🌍 {pais_sel}: No tiene datos registrados para este indicador.")

        # --- SECCIÓN C: TOP 5 RANKING ---
        print("\n" + "-"*60)
        print(f"🏆 Top 5 Países (según filtro de cobertura >{cobertura_min}%):")
        top_5 = df_filtrado.nlargest(5, indicador_sel)[['country', indicador_sel]]
        for i, (idx, row) in enumerate(top_5.iterrows(), 1):
            print(f"   {i}. {row['country']:<25} | {row[indicador_sel]:,.2f}")

# Vincular el evento del botón con la función orquestadora
btn_run.on_click(procesar_consulta)

print("✅ Orquestador vinculado. Ya puedes usar el Panel de Control arriba para consultar datos.")

✅ Orquestador vinculado. Ya puedes usar el Panel de Control arriba para consultar datos.


In [7]:
# ── PASO 6: EXPORTACIÓN Y CIERRE DE SESIÓN ────────────────────────────────

def exportar_datos_procesados(b):
    """Genera un CSV con los datos actualmente visibles en la consulta."""
    with out_panel:
        cfg = get_config()
        # Generamos un nombre dinámico basado en el timestamp para evitar sobreescritura accidental
        timestamp = time.strftime("%Y%m%d-%H%M")
        export_name = f"consulta_wb_{timestamp}.csv"

        try:
            # Exportamos el DataFrame actual (df_wide) pero ordenado por el indicador seleccionado
            indicador_sel = drop_ind.value
            if indicador_sel and indicador_sel != "No hay datos":
                df_export = df_wide.sort_values(by=indicador_sel, ascending=False)
                df_export.to_csv(export_name, index=False, encoding="utf-8-sig")

                print("\n" + "="*60)
                print(f"💾 EXPORTACIÓN EXITOSA")
                print(f"   • Archivo: {export_name}")
                print(f"   • Registros: {len(df_export)}")
                print(f"   • Nota: Puedes encontrarlo en la carpeta de archivos (izquierda).")
                print("="*60)
            else:
                print("❌ No hay un indicador seleccionado para exportar.")
        except Exception as e:
            print(f"❌ Error al exportar: {e}")

# Crear botón de exportación adicional
btn_export = widgets.Button(
    description='Descargar CSV',
    button_style='info',
    icon='download',
    layout={'width': 'max-content'}
)

# Vincular evento
btn_export.on_click(exportar_datos_procesados)

# Mostrar el botón debajo del panel de control previo
display(widgets.HBox([widgets.HTML("<b>Opciones de salida:</b>"), btn_export]))

print("✅ Módulo de exportación activado.")
print("🚀 Aplicación Web completa y funcional.")

✅ Módulo de exportación activado.
🚀 Aplicación Web completa y funcional.


In [10]:
# ── HERRAMIENTA PEDAGÓGICA: RADAR DE DETECCIÓN DE OUTLIERS (INNOVACIÓN) ─────

def detectar_outliers_innovacion(b):
    """
    Detecta países con alto desempeño social pero bajo nivel económico.
    Lógica: PIB Bajo (Percentil < 40) + Resultado Social Alto (Percentil > 70).
    """
    with out_panel:
        clear_output(wait=True)
        cfg = get_config()

        # 1. Recuperar indicadores dinámicamente desde el JSON
        # Buscamos la llave técnica del PIB definida en el Paso 1
        ind_pib = cfg["diagnose_indicators"].get("NY.GDP.PCAP.CD", "PIB per cápita (USD)")
        ind_social = drop_ind.value

        if ind_social == ind_pib or ind_social == "No hay datos":
            print("❌ Error: Selecciona un indicador social en el menú superior para comparar contra el PIB.")
            return

        # 2. Creación de copia limpia para evitar SettingWithCopyWarning
        # Solo procesamos países que tengan ambos datos
        df_radar = df_wide.dropna(subset=[ind_pib, ind_social]).copy()

        if df_radar.empty:
            print("❌ No hay datos suficientes para realizar el cruce de variables.")
            return

        # 3. Cálculo de Rankings (Percentiles) con .loc para seguridad
        df_radar.loc[:, 'rank_pib'] = df_radar[ind_pib].rank(pct=True)
        df_radar.loc[:, 'rank_social'] = df_radar[ind_social].rank(pct=True)

        # Inversión de ranking para indicadores donde 'menos es mejor'
        # (Si es mortalidad o crimen, estar en el 'Top mejor desempeño' significa tener valor bajo)
        if any(k in ind_social.lower() for k in ["mort", "homicidios", "crimen", "muerte"]):
             df_radar.loc[:, 'rank_social'] = 1 - df_radar['rank_social']

        # 4. Filtrado de Outliers (Innovadores)
        # Criterio: 40% más pobres vs 30% con mejor desempeño social
        innovadores = df_radar[
            (df_radar['rank_pib'] < 0.4) & (df_radar['rank_social'] > 0.7)
        ].copy()

        # 5. Visualización de Resultados
        print(f"🚀 RADAR DE INNOVACIÓN: {ind_social.upper()} vs {ind_pib.upper()}")
        print("="*85)
        print("Buscando países que logran resultados excepcionales con recursos limitados...")
        print("-" * 85)

        if innovadores.empty:
            print("🧐 No se detectaron outliers claros. Intenta con un indicador diferente.")
        else:
            header = f"{'PAÍS':<30} | {'ECONOMÍA (Ranking)':<22} | {'DESEMPEÑO SOCIAL'}"
            print(header)
            print("-" * 85)

            # Ordenar por el mejor desempeño social
            for _, row in innovadores.sort_values('rank_social', ascending=False).iterrows():
                pib_pct = int(row['rank_pib'] * 100)
                soc_pct = int(row['rank_social'] * 100)

                # Formateo visual
                linea = f"{row['country'][:28]:<30} | Top {pib_pct:>2}% más pobre     | Top {soc_pct:>2}% mejor nivel"
                print(linea)

            print("\n" + "💡 INSTRUCCIÓN PEDAGÓGICA:")
            print("Estos países están 'venciendo' la estadística. Investiga qué innovación en")
            print("salud, educación o seguridad están aplicando para lograr tanto con tan poco.")

# Configuración del Botón
btn_radar = widgets.Button(
    description='Detectar Outliers (Innovación)',
    button_style='warning', # Color naranja/amarillo
    icon='magic',
    layout={'width': '280px'}
)

btn_radar.on_click(detectar_outliers_innovacion)

# Mostrar herramienta
display(widgets.HTML("<br>"))
display(widgets.HBox([widgets.HTML("<b>🔍 Casos de Estudio:</b>"), btn_radar]))

HTML(value='<br>')